# Lab 3.5 - Deploying and Performing Batch Predictions with Amazon SageMaker

## Overview

This lab is a continuation of the guided labs in Module 3.

In this lab, you will:
- Deploy a trained XGBoost model using Amazon SageMaker.
- Perform predictions against the deployed model.
- Delete the SageMaker endpoint.
- Perform batch transformation on the test dataset.
- Convert probability predictions into binary classifications.
- Experiment with prediction thresholds.

## Business Scenario

You work for a healthcare provider and want to improve the detection of abnormalities in orthopedic patients.

The dataset contains six biomechanical features and a target indicating whether a patient is **Normal** or **Abnormal**.

> **Note:** This lab is for machine-learning experimentation and should not be used as a clinical diagnostic system.

## Dataset

The Vertebral Column dataset contains biomechanical measurements derived from the pelvis and lumbar spine.

The six attributes are:
1. Pelvic incidence
2. Pelvic tilt
3. Lumbar lordosis angle
4. Sacral slope
5. Pelvic radius
6. Grade of spondylolisthesis

For this binary classification task:
- `Normal` → `0`
- `Abnormal` → `1`

The dataset originally contains Normal, Disk Hernia, and Spondylolisthesis categories. Disk Hernia and Spondylolisthesis are combined into the `Abnormal` class.

**Dataset source:** Dua, D. and Graff, C. (2019), UCI Machine Learning Repository.

## Lab Setup

The following cells import the dataset, split it into training, validation, and test sets, upload the data to Amazon S3, and train the XGBoost model.

In [ ]:
# Install required packages
!pip install scipy
!pip install scikit-learn

In [ ]:
# Import required libraries
import warnings
warnings.simplefilter('ignore')

import requests
import zipfile
import io
import os

import pandas as pd
from scipy.io import arff

import boto3
import sagemaker
from sagemaker.image_uris import retrieve
from sklearn.model_selection import train_test_split

In [ ]:
# S3 bucket provided by the lab environment
bucket = 'c221098a5575381l16201290t1w660526816903-labbucket-mduehtu2ysv4'
prefix = 'lab3'

## Importing the Data

In [ ]:
# Download the Vertebral Column dataset
f_zip = 'http://archive.ics.uci.edu/ml/machine-learning-databases/00212/vertebral_column_data.zip'

r = requests.get(f_zip, stream=True)
r.raise_for_status()

Vertebral_zip = zipfile.ZipFile(io.BytesIO(r.content))
Vertebral_zip.extractall()

# Load the ARFF dataset
data = arff.loadarff('column_2C_weka.arff')
df = pd.DataFrame(data[0])

# Convert class labels to binary values
class_mapper = {b'Abnormal': 1, b'Normal': 0}
df['class'] = df['class'].replace(class_mapper)

# Move the target column to the first position
cols = df.columns.tolist()
cols = cols[-1:] + cols[:-1]
df = df[cols]

df.head()

## Create Training, Validation, and Test Sets

In [ ]:
# Split the data into training, validation, and test datasets
train, test_and_validate = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df['class']
)

test, validate = train_test_split(
    test_and_validate,
    test_size=0.5,
    random_state=42,
    stratify=test_and_validate['class']
)

print('Training set:', train.shape)
print('Validation set:', validate.shape)
print('Test set:', test.shape)

In [ ]:
# File names
train_file = 'vertebral_train.csv'
test_file = 'vertebral_test.csv'
validate_file = 'vertebral_validate.csv'

# Create an S3 resource
s3_resource = boto3.Session().resource('s3')

def upload_s3_csv(filename, folder, dataframe):
    csv_buffer = io.StringIO()
    dataframe.to_csv(csv_buffer, header=False, index=False)
    s3_resource.Bucket(bucket).Object(
        os.path.join(prefix, folder, filename)
    ).put(Body=csv_buffer.getvalue())

# Upload datasets to S3
upload_s3_csv(train_file, 'train', train)
upload_s3_csv(test_file, 'test', test)
upload_s3_csv(validate_file, 'validate', validate)

print('Training, validation, and test datasets uploaded to S3.')

## Train the XGBoost Model

In [ ]:
# Retrieve the XGBoost training container
container = retrieve(
    'xgboost',
    boto3.Session().region_name,
    '1.0-1'
)

# Model hyperparameters
hyperparams = {
    'num_round': '42',
    'eval_metric': 'auc',
    'objective': 'binary:logistic'
}

s3_output_location = 's3://{}/{}/output/'.format(bucket, prefix)

# Create the SageMaker estimator
xgb_model = sagemaker.estimator.Estimator(
    container,
    sagemaker.get_execution_role(),
    instance_count=1,
    instance_type='ml.m4.xlarge',
    output_path=s3_output_location,
    hyperparameters=hyperparams,
    sagemaker_session=sagemaker.Session()
)

# Configure training and validation inputs
train_channel = sagemaker.inputs.TrainingInput(
    's3://{}/{}/train/'.format(bucket, prefix),
    content_type='text/csv'
)

validate_channel = sagemaker.inputs.TrainingInput(
    's3://{}/{}/validate/'.format(bucket, prefix),
    content_type='text/csv'
)

data_channels = {
    'train': train_channel,
    'validation': validate_channel
}

# Train the model
xgb_model.fit(inputs=data_channels, logs=False)

print('ready for hosting!')

# Step 1: Hosting the Model

Now that the model has been trained, deploy it using Amazon SageMaker hosting services.

For this lab, a single `ml.m4.xlarge` instance is used.

In [ ]:
# Deploy the trained model
xgb_predictor = xgb_model.deploy(
    initial_instance_count=1,
    serializer=sagemaker.serializers.CSVSerializer(),
    instance_type='ml.m4.xlarge'
)

print('Model deployed successfully.')

# Step 2: Performing Predictions

Review the test data before sending an example to the deployed model.

In [ ]:
# Check the test dataset
print('Test dataset shape:', test.shape)
test.head(5)

In [ ]:
# Select the first row without the target/class column
row = test.iloc[0:1, 1:]
row.head()

In [ ]:
# Convert the row to CSV format
batch_X_csv_buffer = io.StringIO()
row.to_csv(batch_X_csv_buffer, header=False, index=False)

test_row = batch_X_csv_buffer.getvalue()
print(test_row)

In [ ]:
# Perform prediction
prediction = xgb_predictor.predict(test_row)

print(prediction)

### Prediction Result

The XGBoost model uses `binary:logistic`, so the prediction is a probability between 0 and 1 rather than an immediate `0` or `1`.

For the first test row, the provided lab output was approximately:

`0.9966071844100952`

The corresponding actual class is `1` (**Abnormal**), so the prediction is accurate for this row.

In [ ]:
# Compare the first prediction with the actual class
actual_class = test.iloc[0]['class']
predicted_probability = float(prediction.decode('utf-8'))

print('Actual class:', actual_class)
print('Predicted probability:', predicted_probability)
print('Predicted class at threshold 0.5:', int(predicted_probability >= 0.5))
print('Correct:', int(predicted_probability >= 0.5) == actual_class)

## Challenge: Predict the Second Row

The following cell sends the second row to the deployed model. You can change `row_number` to test other rows.

In [ ]:
def predict_row(row_number, threshold=0.5):
    row = test.iloc[row_number:row_number + 1, 1:]

    csv_buffer = io.StringIO()
    row.to_csv(csv_buffer, header=False, index=False)

    prediction = xgb_predictor.predict(csv_buffer.getvalue())
    probability = float(prediction.decode('utf-8'))
    predicted_class = int(probability >= threshold)
    actual_class = int(test.iloc[row_number]['class'])

    return {
        'row': row_number,
        'actual': actual_class,
        'probability': probability,
        'predicted': predicted_class,
        'correct': predicted_class == actual_class
    }

# Second row
predict_row(1)

In [ ]:
# Try several rows
results = pd.DataFrame([predict_row(i) for i in range(10)])
results

# Step 3: Terminating the Deployed Model

After completing the individual predictions, delete the endpoint to stop the hosted SageMaker resource.

In [ ]:
# Delete the SageMaker endpoint
xgb_predictor.delete_endpoint(delete_endpoint_config=True)

print('Endpoint deleted.')

# Step 4: Performing a Batch Transform

Instead of sending test records individually to a deployed endpoint, SageMaker Batch Transform can process the complete test dataset.

SageMaker will:
1. Start an instance containing the model.
2. Process the input records.
3. Write predictions to Amazon S3.
4. Terminate the transform instance.

In [ ]:
# Select all test rows and remove the target column
batch_X = test.iloc[:, 1:]
batch_X.head()

In [ ]:
# Upload the batch input file to S3
batch_X_file = 'batch-in.csv'

upload_s3_csv(
    batch_X_file,
    'batch-in',
    batch_X
)

print('Batch input uploaded.')

In [ ]:
# Configure batch transform
batch_output = 's3://{}/{}/batch-out/'.format(bucket, prefix)
batch_input = 's3://{}/{}/batch-in/{}'.format(
    bucket,
    prefix,
    batch_X_file
)

xgb_transformer = xgb_model.transformer(
    instance_count=1,
    instance_type='ml.m4.xlarge',
    strategy='MultiRecord',
    assemble_with='Line',
    output_path=batch_output
)

In [ ]:
# Start the batch transform job
xgb_transformer.transform(
    data=batch_input,
    data_type='S3Prefix',
    content_type='text/csv',
    split_type='Line'
)

# Wait for the transform job to finish
xgb_transformer.wait()

print('Batch transform completed.')

## Download and Review Batch Predictions

In [ ]:
# Download the batch transform output from S3
s3 = boto3.client('s3')

obj = s3.get_object(
    Bucket=bucket,
    Key='{}/batch-out/{}'.format(
        prefix,
        'batch-in.csv.out'
    )
)

target_predicted = pd.read_csv(
    io.BytesIO(obj['Body'].read()),
    sep=',',
    names=['class']
)

target_predicted.head(5)

## Convert Probabilities into Binary Predictions

The lab uses a threshold of `0.65`.

- Probability > 0.65 → `1` (Abnormal)
- Probability ≤ 0.65 → `0` (Normal)

In [ ]:
def binary_convert(x, threshold=0.65):
    if x > threshold:
        return 1
    else:
        return 0

target_predicted['binary'] = target_predicted['class'].apply(
    lambda x: binary_convert(x, threshold=0.65)
)

print('Predictions:')
display(target_predicted.head(10))

print('Original test data:')
display(test.head(10))

## Challenge: Experiment with the Threshold

Changing the classification threshold can change the final binary predictions.

A lower threshold generally makes it easier for a record to be classified as `Abnormal`, while a higher threshold makes the model more conservative.

In [ ]:
# Compare several classification thresholds
thresholds = [0.30, 0.50, 0.65, 0.80, 0.90]

threshold_results = []

for threshold in thresholds:
    predictions = (
        target_predicted['class'] > threshold
    ).astype(int)

    accuracy = (
        predictions.values == test['class'].values
    ).mean()

    threshold_results.append({
        'threshold': threshold,
        'predicted_abnormal': int(predictions.sum()),
        'accuracy': accuracy
    })

threshold_results = pd.DataFrame(threshold_results)
threshold_results

# Conclusion

The trained XGBoost model was deployed to a SageMaker endpoint and used for individual predictions. The endpoint was then deleted to avoid leaving the hosted resource running.

A SageMaker Batch Transform job was subsequently used to process the complete test dataset. The returned probability scores were converted into binary classifications using a configurable threshold.

The threshold can affect the final predictions and therefore the measured performance of the model. In the following lab, metrics can be generated and the model can be further evaluated and tuned.